---
title: Plotting
execute:
  enabled: true
---

This page's code cells are **executed live** by Quarto (via Jupyter) every time the
docs are built. `q.plot` renders interactive Plotly figures for return streams —
quick column charts, equity curves, drawdowns, calendar heatmaps, and Monte Carlo
diagnostics — with sensible defaults for quant research. Return-stream *statistics*
(performance, alpha/beta, rolling diagnostics, `q.stats.montecarlo`, ...) live in
`q.stats`; see [Return Statistics](stats.ipynb). For feature engineering and
price-chart examples, see [Feature Engineering](feat.ipynb).

## Sample data

We reuse qrt's bundled sample datasets — AAPL as the "strategy" and SPY as the
benchmark — loaded offline via `q.data.datasets.load`, no network dependency:

In [1]:
import pandas as pd

import qrt as q

aapl = q.data.datasets.load("aapl")
spy = q.data.datasets.load("spy")

returns = pd.concat(
    {
        "AAPL": aapl["close"].pct_change(),
        "SPY": spy["close"].pct_change(),
    },
    axis=1,
).dropna()
strategy = returns["AAPL"]
benchmark = returns["SPY"]

returns.tail()

,AAPL,SPY
datetime,,
2026-07-10,-0.002846,0.004310
2026-07-13,0.006311,-0.007656
2026-07-14,-0.007721,0.003551
2026-07-15,0.040145,0.003964
2026-07-16,0.017588,-0.005419


## Quick column charts: `q.plot.col`

`q.plot.col` renders any Series/DataFrame column selection as an interactive line
chart, with shell-style wildcard support (e.g. `"*_ret"`) for selecting multiple
columns at once:

In [2]:
fig = q.plot.col(returns, title="Daily returns", ylabel="Return")
fig.show()

## Equity curve: `q.plot.equity`

Compounds a return stream into a growth-of-$1 curve:

In [3]:
fig = q.plot.equity(strategy, title="AAPL equity curve")
fig.show()

## Drawdown: `q.plot.drawdown`

The underwater chart — cumulative drawdown from the running peak:

In [4]:
fig = q.plot.drawdown(strategy)
fig.show()

## Full performance report: `q.plot.plot`

Combines the equity curve, drawdown, and headline stats (CAGR, Volatility, Sharpe,
Max Drawdown) into one linked-axis figure. An optional `benchmark` overlays a second
equity curve:

In [5]:
fig = q.plot.plot(strategy, benchmark=benchmark, title="AAPL vs. SPY")
fig.show()

## Calendar returns: `q.plot.monthly_heatmap`

Year-by-month compounded returns as an interactive heatmap, pairing with
`q.stats.monthly_returns`:

In [6]:
fig = q.plot.monthly_heatmap(strategy, title="AAPL monthly returns")
fig.show()

## Monte Carlo fan chart: `q.plot.montecarlo`

Runs `q.stats.montecarlo` — which resamples historical returns *with replacement*
(a bootstrap), not a plain permutation, since permuting a fixed set of returns
can't change their compounded terminal value — and renders a sample of the
simulated paths as a fan chart around a shaded confidence band. Paths are colored
by outcome relative to the optional `bust`/`goal` thresholds, and the original
(unresampled) path is highlighted on top.

By default each period is resampled independently (i.i.d.), which assumes no
serial dependence between returns — an assumption real markets violate through
volatility clustering and autocorrelation. Passing `block_size` switches to a
*stationary block bootstrap* that resamples contiguous, randomly-lengthed blocks
of history instead of individual days, preserving that local structure.

We also decouple the simulation horizon from the resampling pool: `returns` is
the full AAPL history (avoiding bias toward a single market regime), but
`periods=252` simulates only one trading year forward. Without this, compounding
variance over the full ~26-year history balloons the terminal-value spread to
unreadable extremes and saturates `bust`/`goal` probabilities near 0%/100%
(a long enough random path is nearly guaranteed to cross any fixed threshold
eventually) — real effects, but a poor, uninformative simulation:

In [7]:
fig = q.plot.montecarlo(
    strategy,
    sims=1000,
    periods=252,
    bust=-0.3,
    goal=0.5,
    seed=7,
    sample=150,
    block_size=20,
    title="AAPL Monte Carlo simulation (1-year horizon, block bootstrap)",
)
fig.show()

## Monte Carlo distribution: `q.plot.montecarlo_distribution`

Complements the fan chart with two histograms built from the same (block
bootstrap, 1-year horizon) simulations: the spread of terminal returns (reward)
alongside the spread of each simulation's Max Drawdown (risk) — surfacing the
downside risk a terminal-value histogram alone would hide. The original outcome
and the `bust`/`goal` thresholds are marked on each panel:

In [8]:
fig = q.plot.montecarlo_distribution(
    strategy,
    sims=1000,
    periods=252,
    bust=-0.3,
    goal=0.5,
    seed=7,
    block_size=20,
    title="AAPL Monte Carlo outcome distribution (1-year horizon, block bootstrap)",
)
fig.show()

## Saving figures: `q.plot.show`

Any figure returned above can be displayed and/or exported as self-contained HTML
and/or PNG (the latter requires `kaleido`) via `q.plot.show`. This example writes
files, so it's shown but not executed here:

```python
q.plot.show(fig, "montecarlo", save_to="exports", formats=["html", "png"])
```